# Workplace choice: where a worker works

## What this model answers

Which zone does a worker work in? The model compares the chosen workplace zone with 99 zones drawn at random by the number of jobs, car time from home, boundaries and the job mix, with time effects that differ by person.

| | |
|---|---|
| Choice | The workplace zone |
| Alternatives | The chosen zone and 99 zones drawn at random from the 1,587 zones with jobs (hold-out: all 1,587) |
| Decision makers | Workers with a fixed usual workplace |
| Data | Regional Travel Survey 2017/18 (Baltimore and Washington regions) |
| Estimation sample | 9,707 workers |

## How to run

- Kernel: *Python (biogeme)* (`~/.venvs/biogeme`, Biogeme 3.3); the data rebuild also needs `python3` with geopandas. Then *Run All*.
- `RUN_DATA_PREP`: rebuild the estimation tables in a scratch folder and compare them with the shipped ones.
- `RUN_CHECK`: log-likelihood at the article coefficients, no estimation (about 0.2 min).
- `RUN_ESTIMATION`: run the Biogeme estimation (off by default; the results in `output/` are read instead).
- `RUN_HOLDOUT`: log-likelihood of the hold-out records.
- Figures of the hold-out check are read from `output/holdout_*` (copied there by the notebook builder).

## Contents

Article table: `est_workplace`. This notebook goes from the raw records to the estimates of this model:

1. Raw data
2. Data preparation: filters from the raw records to the estimation sample (records kept and dropped), rebuild check
3. Estimation data and choice set
4. Descriptive statistics and figures; 4.1 every explanatory variable of the model in the data
5. Model specification (the folder's `model/estimate_workplace_choice.py`, unchanged)
6. Correlation of the explanatory variables (heatmaps, VIF)
7. Log-likelihood at the article coefficients
8. Biogeme estimation
9. Results: article format, fit, comparison with the article, correlation of the estimates
10. Hold-out sample

11. Glossary

## Model at a glance

**1. Who decides.** Workers with a fixed usual workplace. Data: Regional Travel Survey 2017/18 (Baltimore and Washington regions). Estimation sample: 9,707 workers.

**2. What is chosen (dependent variable).** The workplace zone. Alternatives: The chosen zone and 99 zones drawn at random from the 1,587 zones with jobs (hold-out: all 1,587).

**3. Explanatory variables** (signs of the article estimates, `model/reference_estimates.csv`):

| Group | Variable | What it measures | Sign |
|---|---|---|---|
| Size | Size coefficient mu | ln of the zone's jobs | + |
| Car time | Car time (min) | AM-peak car minutes home to zone | − |
| Car time | ln(1 + car time) | flattens the time effect | − |
| Car time x person | Time x woman | Time x woman | − (shift for women) |
| Car time x person | Time x age under 30 | Time x age under 30 | − (shift for workers under 30) |
| Car time x person | Time x age 55+ | Time x age 55+ | − (shift for workers 55 and older) |
| Car time x person | Time x income under $40k | Time x income under $40k | − (shift for incomes under $40k) |
| Car time x person | Time x income $75-125k | Time x income $75-125k | + (shift for incomes of $75-125k) |
| Car time x person | Time x income $125k+ | Time x income $125k+ | + (shift for incomes of $125k or more) |
| Car time x person | Time x driving licence | Time x driving licence | + (shift for licence holders) |
| Car time x person | Time x household size | per person | − (shift for each extra household member) |
| Car time x person | Time x rural home | Time x rural home | + (shift for workers with a rural home zone) |
| Car time x person | Time x no car | Time x no car | − (shift for workers in households without a car) |
| Borders and the District | Same zone | workplace in the home zone | + |
| Borders and the District | Other county | workplace in another county | − |
| Borders and the District | Other state | workplace in another state | − |
| Borders and the District | DC workplace, home outside DC | District workplace for non-residents | + |
| Job mix x person | Office share x income under $40k | office share of the zone's jobs | − |
| Job mix x person | Retail share x age under 30 | retail share of the zone's jobs | + |
| Job mix x person | Industrial share x woman | industrial share of the zone's jobs | − |
| Wage proxy | High-wage share | share of the zone's jobs with high wages | + |
| Wage proxy | High-wage share x income under $40k | High-wage share x income under $40k | − (shift for incomes under $40k) |
| Wage proxy | High-wage share x income $75-125k | High-wage share x income $75-125k | + (shift for incomes of $75-125k) |
| Wage proxy | High-wage share x income $125k+ | High-wage share x income $125k+ | + (shift for incomes of $125k or more) |

**4. How household traits enter.** Household traits (income, size, age, cars, children) are the same for every alternative, so on their own they cannot change which alternative is chosen. They enter only as interactions with attributes of the alternatives (for example density x single-person household, car time x income).

**5. How to read a coefficient.** A positive coefficient makes the alternative more likely, a negative one less likely. exp(coefficient) is the factor by which the odds change for one unit more of the variable.
 Income enters as ln(income / $50k), so a main coefficient is that of a $50k household and the x ln income term shifts it: Car time (min) is −0.08 at $25k, −0.07 at $50k and −0.06 at $100k.

In [ ]:
# switches (see How to run) and set-up: imports, folders, helper modules
RUN_DATA_PREP  = True    # rebuild the estimation sample and tables into output/rebuild_check and compare with the shipped tables (section 2)
RUN_CHECK      = True    # log-likelihood at the article coefficients
RUN_ESTIMATION = False   # Biogeme estimation, written to output/ (False: results are read from output/ if there are any)
RUN_HOLDOUT    = False    # log-likelihood of the hold-out sample (at your estimates if there are any, else at the article coefficients)

import os, sys, json, glob, inspect, importlib, time, re
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import biogeme.biogeme as bio
from biogeme.parameters import Parameters
from biogeme.results_processing import EstimationResults

FOLDER = os.path.abspath("")                       # this model folder (VS Code starts the notebook here)
sys.path.insert(0, os.path.join(FOLDER, "..", "..", "tools")); import nbtools as nt; import nb_reader as rd; import figs as fg
fg = importlib.reload(fg)
from IPython.display import Markdown
sys.path.insert(0, os.path.join(FOLDER, "model"))
RUN = json.load(open(os.path.join(FOLDER, "RUN.json")))
OUT = os.path.join(FOLDER, "output"); os.makedirs(OUT, exist_ok=True)
NAME = "workplace_choice"; MODEL_FOLDER = "05_workplace_choice"
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("model folder:", os.path.basename(FOLDER))

## 1. Raw data

*Where the records come from. Look for: the files and the record counts.*

In [ ]:
# Raw data
SH = os.path.abspath(os.path.join(FOLDER, "..", "00_shared_data")); CODE = os.path.join(FOLDER, "input", "code")
RAW = os.path.join(FOLDER, "input", "data", "raw"); REBUILD = os.path.join(OUT, "rebuild_check")
ZS = pd.read_csv(os.path.join(SH, "zone_system", "zoneSystem.csv"))[["ZoneId", "COUNTYFIPS", "JUR_NAME", "PUMA_10"]].rename(columns={"ZoneId": "zone"})
ZS["area_group"] = nt.area_group_of_county(ZS.COUNTYFIPS)

def rebuild_tables(python=sys.executable, pre=""):
    """build_estimation_data.main() of input/code with its output folder set to a scratch folder; returns the comparison with the
    shipped estimation and hold-out tables."""
    out = os.path.join(REBUILD, "data"); [os.makedirs(os.path.join(out, h), exist_ok=True) for h in ("estimation", "holdout")]
    nt.run(f'{python} -c "import build_estimation_data as B; {pre}B.OUT = {out!r}; B.main()"', cwd=CODE)
    return pd.concat([nt.compare_trees(os.path.join(out, h), os.path.join(FOLDER, "input", "data", h)).assign(half=h) for h in ("estimation", "holdout")],
                     ignore_index=True)

### 1.1 The 2017/18 Regional Travel Survey (workers)

The workplace model uses the workers of the 2017/18 RTS with a fixed usual workplace. Their home and workplace locations
are the survey traffic zones (BMC, TPB) or tracts, spread over the model zones with housing-unit (home) and job (workplace)
weights; the worker file of `input/data/raw` holds one row per worker with the modal home zone `hz` and workplace zone `wz`.

In [ ]:
# 1.1 The 2017/18 Regional Travel Survey (workers)
flat = pd.read_csv(os.path.join(SH, "rts_2017_18", "RTS_raw_flat_data.csv"), low_memory=False, usecols=lambda c: not nt.RACE.search(c))
per = flat.drop_duplicates("person_id")
WK = pd.read_parquet(os.path.join(RAW, "workers_v2.parquet"))
print(f"persons of the survey: {len(per):,}; workers in workers_v2.parquet: {len(WK):,} (estimation {int((WK.half == 'estimation').sum()):,})")
WK.head()

## 2. Data preparation

*From the raw records to the estimation sample. Look for: the records kept and dropped at each filter. With RUN_DATA_PREP the rebuilt tables must equal the shipped ones.*

### 2.1 From survey persons to workers

The worker script of the shared DAYMAP preparation (`../../ABIT/00_shared_prep/code`, step S15) keeps the persons with a fixed usual workplace and a work tract, locates
home and workplace in the zone system (traffic-zone crosswalks, else tract splits) and drops the records below. With
`RUN_DATA_PREP` it is run into a scratch folder (with the shipped crosswalks and tract splits) and its record log is shown.

In [ ]:
# Data preparation
n_fixed = int(((per.j1_workplace_loc == 1) & (per.work_tract_fips > 0)).sum())
print(f"persons with a fixed usual workplace and a work tract code (mirrored): {n_fixed:,}")
if RUN_DATA_PREP:
    import shutil
    RW = os.path.join(REBUILD, "abit_shared"); os.makedirs(os.path.join(RW, "taz_crosswalks"), exist_ok=True)
    for f_ in os.listdir(os.path.join(nt.ABIT_SHARED, "taz_crosswalks")):
        shutil.copy2(os.path.join(nt.ABIT_SHARED, "taz_crosswalks", f_), os.path.join(RW, "taz_crosswalks"))
    nt.run("python3 rts_11_data_v2.py", cwd=nt.ABIT_PREP, env={"ABIT_REBUILT": RW}, quiet=True)
    lg = pd.read_csv(os.path.join(RW, "taz_crosswalks", "tables_rts", "drop_log_v2.csv"))
    st = []; n = None
    for step, v in zip(lg.step, lg.n):
        n = n + v if step.startswith("dropped") else v
        st.append((step.replace("dropped: ", "without: "), n))
    display(nt.funnel2({"both halves": st[:-2]}))
    print(dict(st[-2:]))
    res_w = nt.compare_frames(pd.read_parquet(os.path.join(RW, "taz_crosswalks", "workers_v2.parquet")), WK, name="workers_v2.parquet")
    display(res_w)
else:
    print("skipped (RUN_DATA_PREP = False)")

### 2.2 Rebuild check of the model data

`w01_data.py` (zone data, car skim, workers: `data.npz`) runs with its output redirected (`W01_OUT`), then `build_estimation_data.main()` (the 99 sampled alternatives of each worker, seed of the original run) with a scratch output folder; both are compared with the shipped files.

In [ ]:
# 2.2 Rebuild check of the model data
if RUN_DATA_PREP:
    t0 = time.time(); npz = os.path.join(REBUILD, "data.npz")
    nt.run(f"{sys.executable} w01_data.py", cwd=CODE, env={"W01_OUT": npz}, quiet=True)
    a, b = np.load(npz, allow_pickle=True), np.load(os.path.join(RAW, "data.npz"), allow_pickle=True)
    eq = lambda u, v: u.shape == v.shape and (np.array_equal(u, v, equal_nan=True) if u.dtype.kind in "fc" else np.array_equal(u, v))
    same = {k_: k_ in a.files and eq(a[k_], b[k_]) for k_ in b.files}
    bad = [k_ for k_, v in same.items() if not v]
    res_n = pd.DataFrame([{"table": f"data.npz ({len(same)} arrays)", "result": "identical" if not bad else "DIFFERENT", "differing columns": ", ".join(bad)}])
    res = rebuild_tables(); print(f"rebuild: {(time.time() - t0) / 60:.1f} min")
    res = pd.concat([res_w, res_n, res], ignore_index=True); display(res)
    nt.rebuild_verdict(res, glob.glob(os.path.join(FOLDER, "model", "estimate_*.py"))[0])
else:
    print("skipped (RUN_DATA_PREP = False)")

### Data sources

Where each source comes from and where it goes (paths from `manual_estimation/`). *verify*: no download link is recorded in the project. The full download steps are in `INSTRUCTIONS.pdf`.

**Regional Travel Survey 2017/18 (household, person, trip records)** — Baltimore Metropolitan Council (BMC) and National Capital Region Transportation Planning Board (TPB/MWCOG)
- Where: https://www.baltometro.org
- How: Restricted: request the 2017/2018 Regional Travel Survey data from BMC / TPB. File used: RTS_raw_flat_data.csv (flat person-level file with household fields: household_id, person_id, home_state_county_fips, home_tract_fips, home_type, hhsize, numvehicle, numdrivers, hh_income_detailed, wthhfin, wtperfin, age, age_group, employment_status, j1_workplace_loc, work_state_county_fips, work_tract_fips, license, school locations and BMC/TPB TAZ codes). Also rts_trips_clean.csv (cleaned trips) for the synthetic-population workplace step.
- Put in: 00_shared_data/raw/rts_2017_18/
- Files: `RTS_raw_flat_data.csv`, `rts_trips_clean.csv`

**MSTM zone system, zone polygons and peak car skim (Maryland Statewide Transportation Model)** — Maryland Department of Transportation State Highway Administration (MDOT SHA), provided to the project
- Where: https://roads.maryland.gov
- How: Not a public download: the 1,588-zone MSTM system (zoneSystem.csv), zone polygons (SMZ_RMZ_02152011inMSTM_EPSG26918.shp) and the AM-peak auto travel-time skim were provided by MDOT. Request them from MDOT SHA (MSTM / travel forecasting).
- Put in: 00_shared_data/raw/mstm/
- Files: `zoneSystem.csv`, `SMZ_RMZ_02152011inMSTM_EPSG26918.shp/.shx/.dbf/.prj`, `auto_peak_time.csv.gz`

**LEHD LODES 8, Workplace Area Characteristics 2016 and geography crosswalk, Maryland** — U.S. Census Bureau, LEHD
- Where: https://lehd.ces.census.gov/data/lodes/LODES8/md/wac/md_wac_S000_JT00_2016.csv.gz
- How: Download md_wac_S000_JT00_2016.csv.gz and https://lehd.ces.census.gov/data/lodes/LODES8/md/md_xwalk.csv.gz (other states: .../LODES8/<st>/wac/<st>_wac_S000_JT00_<year>.csv.gz). Workplace zone attributes (high-earning share CE03/C000) use the 2017 WAC files of md, dc, va, pa, de, wv: https://lehd.ces.census.gov/data/lodes/LODES8/<st>/wac/<st>_wac_S000_JT00_2017.csv.gz (project: data/raw/lodes_wac/ and data/raw/lodes_wac_2017_extra/).
- Put in: 00_shared_data/raw/lodes_wac/
- Files: `md_wac_S000_JT00_2016.csv.gz`, `md_xwalk.csv.gz`, `<st>_wac_S000_JT00_2017.csv.gz (st = md, dc, va, pa, de, wv)`

**BLS JOLTS job-openings rate, Maryland** — U.S. Bureau of Labor Statistics
- Where: https://api.bls.gov/publicAPI/v1/timeseries/data/JTS000000240000000JOR
- How: BLS public API v1, series JTS000000240000000JOR (Maryland) and JTS000000000000000JOR (U.S.), annual mean of monthly SA values.
- Put in: 00_shared_data/raw/jolts/
- Files: `jolts_bls_api.json`

**Census 2010 Redistricting (PL 94-171) Maryland file and TIGER 2010 blocks and tracts** — U.S. Census Bureau
- Where: https://www2.census.gov/census_2010/01-Redistricting_File--PL_94-171/Maryland/
- How: md2010.pl.zip (geo file and segment 2, housing units H1) and TIGER/Line 2010 tl_2010_24_tabblock10.zip; tracts tl_2010_<st>_tract10.zip for st = 10, 11, 24, 42, 51, 54 (https://www2.census.gov/geo/tiger/TIGER2010/TRACT/2010/).
- Put in: 00_shared_data/raw/census2010/
- Files: `md2010.pl.zip`, `tl_2010_24_tabblock10.zip`, `tl_2010_<st>_tract10.zip`

## 3. Estimation data and choice set

*The table the model reads. Look for: the columns used and what they mean (data dictionary).*

Alternatives: the chosen workplace zone (alternative 0) and 99 zones drawn uniformly from the other zones with jobs (`zone_<j>`, car time `t_<j>`, job mix and region of the zone). The hold-out is evaluated on all 1,587 zones with jobs.

The table exactly as used in the estimation, and its data dictionary.

In [ ]:
# Estimation data and choice set
est = os.path.join(FOLDER, "input", "data", "estimation")
display(nt.files_table(est))
dd = pd.read_csv(os.path.join(est, "data_dictionary.csv"))
dd

In [ ]:
# Estimation data and choice set
import estimate_workplace_choice as M
df = pd.read_parquet(M.DATA)
print(f"{df.shape[0]:,} rows (decision makers), {df.shape[1]:,} columns")
df.head()

## 4. Descriptive statistics

*What the sample looks like. Look for: plausible shares and maps, and the chosen alternatives against the available ones.*

In [ ]:
# Workplace area by home area (row %), car time to the chosen workplace against the 99 sampled zones (uniform ov
HO = pd.read_csv(os.path.join(FOLDER, "input", "data", "holdout", "workplace_holdout_workers.csv"))
ag = pd.concat({"home": nt.half_comparison(nt.area_group_of_zone(df.home_zone).astype(str), nt.area_group_of_zone(HO.home_zone).astype(str), "x"),
                "workplace": nt.half_comparison(nt.area_group_of_zone(df.work_zone).astype(str), nt.area_group_of_zone(HO.work_zone).astype(str), "x")})
od = pd.crosstab(nt.area_group_of_zone(df.home_zone), nt.area_group_of_zone(df.work_zone), normalize="index").mul(100).round(1)
od.index.name = "home area"; od.columns.name = "workplace area"
display(od)
fig, ax = nt.panels(3, ncol=3, w=4.4, h=2.9)
nt.hist(df.t_0.clip(upper=120), "car time home - workplace (min, AM peak)", "workers", bins=48, ax=ax[0])
b = nt.binned(df.t_0, [0, 10, 20, 30, 45, 60, 90, 1e9], ["<10", "10-20", "20-30", "30-45", "45-60", "60-90", "90+"]); nt.bar(b, "car time to work (min)", "share of workers (%)", ax=ax[1])
nt.hist(df[[f"t_{j}" for j in range(1, 100)]].to_numpy().ravel().clip(max=240), "car time to the sampled alternatives (min)", "alternatives", bins=48, ax=ax[2], color=nt.LIGHT)
plt.tight_layout(); plt.show()
print(f"median car time to work {df.t_0.median():.1f} min (sampled alternatives {np.nanmedian(df[[f't_{j}' for j in range(1, 100)]].to_numpy()):.1f} min); same zone: {100 * (df.home_zone == df.work_zone).mean():.1f} %")

Workplace area by home area (row %), car time to the chosen workplace against the 99 sampled zones (uniform over the zones with jobs).

In [ ]:
# Workers (income group, cars, age, sex, licence, household size) and the job attraction of the zones
fig, ax = nt.panels(6, ncol=3, w=4.3, h=2.8)
nt.bar((100 * df.income_group.value_counts(normalize=True)).sort_index(), "income group (1 < 40k ... 4 125k+)", "share (%)", ax=ax[0])
nt.bar((100 * df.car_seg.map({0: "no car", 1: "cars < workers", 2: "cars >= workers"}).value_counts(normalize=True)), "cars", "share (%)", ax=ax[1], rot=20)
nt.hist(df.age, "age (years)", "workers", bins=range(15, 86, 5), ax=ax[2])
nt.bar((100 * df.female.value_counts(normalize=True)).sort_index(), "female (1 yes)", "share (%)", ax=ax[3])
nt.bar((100 * df.lic.value_counts(normalize=True)).sort_index(), "driving licence (1 yes)", "share (%)", ax=ax[4])
nt.bar((100 * df.hhs.value_counts(normalize=True)).sort_index(), "household size variable hhs", "share (%)", ax=ax[5])
plt.tight_layout(); plt.show()
jz = pd.read_csv(os.path.join(FOLDER, "input", "data", "holdout", "zones_with_jobs.csv"))
fig, ax = nt.panels(3, ncol=3, w=4.4, h=2.9)
jz["jobs"] = jz[["jobs_RET", "jobs_OFF", "jobs_IND", "jobs_OTH"]].sum(axis=1)
nt.hist(np.log1p(jz.jobs), "ln(1 + jobs) of a zone", "zones", bins=40, ax=ax[0])
nt.hist(100 * jz.z_off_sh, "office share of the jobs of a zone (%)", "zones", bins=40, ax=ax[1])
nt.hist(np.log1p(df.jobs_OFF_0 + df.jobs_RET_0 + df.jobs_IND_0 + df.jobs_OTH_0), "ln(1 + jobs) of the chosen zone", "workers", bins=40, ax=ax[2])
plt.tight_layout(); plt.show()

Workers (income group, cars, age, sex, licence, household size) and the job attraction of the zones.

In [ ]:
# Home and workplace zones of the workers, and the area shares of the two halves (the hold-out is evaluated on a
fig, ax = plt.subplots(1, 2, figsize=(13, 5.6))
nt.zone_map(df.home_zone.value_counts(), "workers (home zone)", ax=ax[0], log=True)
nt.zone_map(df.work_zone.value_counts(), "workers (workplace zone)", ax=ax[1], log=True)
ax[0].set_title("homes", fontsize=10); ax[1].set_title("workplaces", fontsize=10)
plt.tight_layout(); plt.show()
ag

Home and workplace zones of the workers, and the area shares of the two halves (the hold-out is evaluated on all 1,587 zones with jobs).

In [ ]:
# Descriptive statistics
alts = nt.alt_columns(df)          # alternative-specific columns <variable>_<j>, used in section 6

### 4.1 Variables in the data

Each explanatory variable of the final specification as it enters the utility, interactions included (read from the model's own utility functions: the part of the utility that each coefficient multiplies; coefficients that share a label, e.g. the copies of a term for several purposes, are shown in one panel). Here: the chosen workplace zone against the 99 sampled zones; unweighted. The car-time interactions use centred person variables, as in the model: car time x (person variable minus its mean).

In [ ]:
# every explanatory variable of the specification: chosen against sampled alternatives
import varplots as vp
_db, _ll = M.model(); U_, AV_, CH_ = vp.find_choice(_ll)
VP = vp.location_from_util(U_, AV_, CH_, df, vp.silo_labels(MODEL_FOLDER), sampled=True)
print(f"{len(VP)} variables, {len(df):,} workers")
vp.location_figures(VP, what='sampled', unit='workers')

How to read it: dark grey line = the chosen alternatives, light bars = the sampled alternatives, as a share of each (%); dashed lines are the two means; 0/1 variables are shown as the share with value 1. A continuous variable that is zero at every alternative of a record cannot affect that record's choice, so such records are left out of its panel (e.g. an interaction with a person group; the share kept is in the panel title). The further apart the two, the more the variable separates the choices.

## 5. Model specification

*The utility functions, read from the model script; nothing is re-specified here.*

The specification is the folder's script `model/estimate_workplace_choice.py`, imported as it is. The start values are the article estimates.

In [ ]:
# Model specification
print(inspect.getsource(M.model))

In [ ]:
# Model specification
t0 = time.time()
database, ll = M.model()
used = nt.model_variables(ll)
print(f"database: {database.num_rows():,} rows; {len(used)} data columns used by the model; built in {time.time()-t0:.1f} s")

## 6. Correlation of the explanatory variables

*Correlation of the explanatory variables. Look for: |r| of 0.7 or more and VIF of 10 or more; these make single coefficients less precise.*

Correlations of the variables that enter the utility, computed over the alternatives available to each decision maker
(alternative-specific variables) and over decision makers (person and household variables). |r| ≥ 0.7 and VIF ≥ 10 are
listed. Interaction terms are correlated with their main effects by construction.

In [ ]:
# Correlation of the explanatory variables
stems = sorted({c.rsplit("_", 1)[0] for c in used if c.rsplit("_", 1)[0] in alts and c.rsplit("_", 1)[-1].isdigit()}
               - {"av", "chosen_set", "zone", "stype"})
long = nt.explanatory(nt.stack_alternatives(df, stems))
print(f"{len(stems)} alternative-specific variables, {len(long):,} available alternatives")
Ca = nt.corr_heatmap(long, "Alternative-specific variables (available alternatives)")
display(nt.high_pairs(Ca)); display(nt.vif(long))
plain = [c for c in used if c in df.columns and c not in sum(alts.values(), []) and c not in ("choice", "weight")]
if len(plain) > 1:
    Xp = nt.explanatory(df[plain]); Cp = nt.corr_heatmap(Xp, "Decision-maker variables"); display(nt.high_pairs(Cp))

## 7. Log-likelihood at the article coefficients

*Log-likelihood at the article coefficients, without estimation. Look for: the two numbers should be equal (to rounding).*

No estimation: the log-likelihood at the start values (the article estimates). Target: article -21376.54,
Biogeme -21376.5448 (N = 9707, K = 24).

In [ ]:
# Log-likelihood at the article coefficients
if RUN_CHECK:
    p = Parameters(); p.set_value("generate_html", False); p.set_value("generate_yaml", False)
    here = os.getcwd(); os.chdir(__import__("tempfile").mkdtemp())
    t0 = time.time()
    b = bio.BIOGEME(database, ll, parameters=p); b.model_name = NAME + "_check"
    ll0 = b.calculate_init_likelihood()
    os.chdir(here)
    print(f"N {database.num_rows():,}   LL at the article coefficients: {ll0:,.4f}   (target -21376.5448)   {time.time()-t0:.0f} s")
else:
    print("skipped (RUN_CHECK = False)")

## 8. Biogeme estimation

*Biogeme estimation, only with RUN_ESTIMATION. Look for: convergence and the run time.*

The same call as `model/estimate_workplace_choice.py`: Biogeme with its defaults (read from `output/biogeme.toml`), start values = article estimates.
Report written to `output/workplace_choice.html`, `.yaml`, and `output/workplace_choice_estimates.csv`.

In [ ]:
# Biogeme estimation
if RUN_ESTIMATION:
    here = os.getcwd(); os.chdir(OUT)
    t0 = time.time()
    b = bio.BIOGEME(database, ll); b.model_name = NAME
    results = b.estimate()
    os.chdir(here)
    print(f"estimation time {time.time()-t0:.0f} s"); print(results.short_summary())
    nt.estimates_table(results)[["parameter", "coefficient", "robust_se", "t", "p"]].rename(
        columns={"coefficient": "Value", "robust_se": "Robust std err.", "t": "Robust t-stat.", "p": "Robust p-value", "parameter": "Name"}
    ).to_csv(os.path.join(OUT, f"{NAME}_estimates.csv"), index=False)
else:
    ref_time = os.path.getmtime(os.path.join(FOLDER, "model", "reference_estimates.csv"))
    y = [f for f in sorted(glob.glob(os.path.join(OUT, f"{NAME}*.yaml")), key=os.path.getmtime) if os.path.getmtime(f) > ref_time and "_check" not in f]
    results = EstimationResults.from_yaml_file(y[-1]) if y else None
    print(f"RUN_ESTIMATION = False: read {os.path.basename(y[-1])} (newer than model/reference_estimates.csv)" if y else "RUN_ESTIMATION = False and no current result in output/: sections 9-10 are skipped")

## 9. Results

*The estimates and what they mean. Look for: coefficients equal to the article within rounding, and the readings below.*

### Estimates in the article format
Coefficient (stars: *** p < 0.01, ** p < 0.05, * p < 0.10, as in the article), robust s.e., p.

In [ ]:
# Results
if results is not None:
    est_t = nt.estimates_table(results)
    display(nt.article_format(est_t))

### What the coefficients say

One plain line per term, from your estimates (from the article estimates if there are none). Income interactions are given at $25k, $50k and $100k; odds change = exp(coefficient) per unit.

In [ ]:
# plain readings of the coefficients and the coefficient plot (95 % robust intervals, grouped by block)
est_v = est_t if results is not None else rd.reference(MODEL_FOLDER)[0]
display(rd.readings(MODEL_FOLDER, est_v).style.hide(axis="index").set_properties(**{"text-align": "left"}))
fg.coef_figure(fg.coef_rows(MODEL_FOLDER, est_v)); plt.show()

How to read it: a point right of zero makes the alternative more likely; an interval that crosses zero is not significant.

### Effects by income

Main term plus its income interaction, with 95 % delta-method bands from the robust covariance.

In [ ]:
# effect on utility by household income (or income group), 95 % delta-method bands
b = results.get_beta_values() if results is not None else dict(zip(est_v.parameter, est_v.coefficient))
names_ = list(b); V = pd.DataFrame(np.asarray(results.robust_variance_covariance_matrix), index=names_, columns=names_) if results is not None else None
fg.group_effect_figure(MODEL_FOLDER, b, V); plt.show()

How to read it: the coefficient for each income group (main term plus its interaction) with a 95 % interval; where the interval crosses zero the effect is not significant for that group.

### Goodness of fit

In [ ]:
# Goodness of fit
if results is not None:
    display(nt.fit_statistics(results))

### Comparison with the article (`model/reference_estimates.csv`)

In [ ]:
# Comparison with the article (model/reference_estimates.csv)
if results is not None:
    REF = os.path.join(FOLDER, "model", "reference_estimates.csv")
    cmp = nt.compare(est_t, REF)
    cmp.to_csv(os.path.join(OUT, "comparison_with_article_notebook.csv"), index=False)
    print(f"{int(cmp.matches_article.sum())} of {len(cmp)} parameters match the article (coefficient and robust s.e. within the 4th-decimal rounding); "
          f"largest |difference|: coefficient {cmp.diff_coef.abs().max():.5f}, s.e. {cmp.diff_se.abs().max():.5f}")
    display(cmp.round({"coefficient": 4, "robust_se": 4, "p": 4, "diff_coef": 5, "diff_se": 5}))
    stats = nt.article_statistics(REF)
    if len(stats):
        stats["Biogeme"] = stats["statistic"].map({"LL(beta)": results.final_loglikelihood, "N": database.num_rows() if "database" in dir() else None,
                                                   "K": results.number_of_free_parameters})
        display(stats)
    nt.plot_against_article(cmp, RUN.get("model", NAME))

### Correlation of the estimates
Robust correlation matrix of the estimated parameters; pairs with |r| ≥ 0.7 listed.

In [ ]:
# Correlation of the estimates
if results is not None:
    display(nt.parameter_correlation(results, "Robust correlation of the estimates"))

## 10. Hold-out sample

*Fit on the hold-out records, which the estimation did not use. Look for: a log-likelihood per record close to that of the estimation sample, and the maps.*

The hold-out of this model is stored in another layout (the full choice set; see `input/data/holdout` and `model/README.md`); its log-likelihood is computed by the project's validation scripts, not in this notebook.

### Hold-out check by area

Observed and model shares of the hold-out records by area (model share = sum of the predicted probabilities). The map and its tables are in `output/holdout_map.png`, `output/holdout_puma.csv`, `output/holdout_county.csv`.

In [ ]:
# hold-out map (copied into output/) and its reading, numbers from output/holdout_*.csv
nt.show_png(os.path.join(OUT, "holdout_map.png"))
display(Markdown(rd.holdout_md(MODEL_FOLDER, OUT, os.path.join(FOLDER, "..", "00_shared_data", "zone_system", "zoneSystem.csv"))))

In [ ]:
# observed against model share by county (hold-out); the counties with the largest differences are labelled
HC = pd.read_csv(os.path.join(OUT, "holdout_county.csv")); u = "Workers"
nt.holdout_scatter(HC, names=rd.county_names(os.path.join(FOLDER, "..", "00_shared_data", "zone_system", "zoneSystem.csv")), unit=u, title="(a) Counties, hold-out")

How to read it: one point per county; on the line the share is predicted exactly, above the line the model is too high.

## Glossary

- **zone**: A traffic analysis zone of the model (1,588 zones).
- **sampled alternatives**: The chosen alternative and others drawn at random with equal probability; this gives the same estimates as the full set, with less precision.
- **size term**: mu x ln(size of the zone): with mu = 1 the probability is proportional to the zone's jobs or enrolment.
- **log-likelihood**: Sum over decision makers of ln(probability of the observed choice). Closer to 0 is better.
- **BIC**: Bayesian information criterion: -2 log-likelihood + K ln N. Lower is better; it penalises extra parameters.
- **hold-out**: The 20 % (residential, cars) or half (survey models) of the records not used for estimation. It was also used in the specification search, so its fit is not a fully independent test.
- **robust s.e.**: Standard error from the sandwich formula, valid when the model is not exactly right.
- **VIF**: Variance inflation factor: how much the correlation with other variables inflates a coefficient's variance; 10 or more is high.